# Check sub-SOCO ⊂ SOCO
Strategia: (1) match esatto via SHA1 dei byte (se i file sono solo rinominati), (2) fallback con pHash + stessa dimensione per i residui, (3) differenza insiemistica per trovare le immagini SOCO non presenti nel subset.

In [1]:
# !pip install pillow imagehash tqdm numpy pandas
from pathlib import Path
import hashlib, pickle
import numpy as np, pandas as pd
from PIL import Image
import imagehash
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor

ROOT = Path("datasets")          # adatta se il notebook sta altrove
SOCO = ROOT / "SOCO"
SUB  = ROOT / "sub-SOCO"
CACHE = ROOT / "hash_cache.pkl"
HASH_SIZE = 16                   # 256 bit
PHASH_THR = 10                   # soglia Hamming (su 256 bit), da tarare guardando l'istogramma
EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

In [2]:
def list_images(base):
    return sorted(p for p in tqdm(base.rglob("*"), desc=f"scan {base.name}") if p.suffix in EXTS)

soco_files = list_images(SOCO / "Images")
sub_files  = list_images(SUB / "images")
print(len(soco_files), "immagini SOCO;", len(sub_files), "immagini sub-SOCO")
print("esempi SOCO:", [str(p.relative_to(SOCO)) for p in soco_files[:3]])
print("esempi sub :", [str(p.relative_to(SUB)) for p in sub_files[:3]])

# struttura annotazioni (utile per ricostruire coppie/split originali)
for d in sorted((SOCO / "PairAnnotations").iterdir()):
    items = sorted(d.iterdir())[:3] if d.is_dir() else [d]
    print(d.name, "->", [i.name for i in items])

scan Images: 0it [00:00, ?it/s]

scan images: 0it [00:00, ?it/s]

4000 immagini SOCO; 646 immagini sub-SOCO
esempi SOCO: ['Images/aeroplane/aeroplane_1.JPEG', 'Images/aeroplane/aeroplane_10.JPEG', 'Images/aeroplane/aeroplane_11.JPEG']
esempi sub : ['images/test/img_0000.jpeg', 'images/test/img_0002.jpeg', 'images/test/img_0009.jpeg']
cross -> ['aeroplane', 'ambulance', 'american_black_bear']
intra -> ['aeroplane', 'ambulance', 'american_black_bear']
trainsplits -> ['test', 'train']


In [3]:
def describe(path):
    path = Path(path)
    data = path.read_bytes()
    sha = hashlib.sha1(data).hexdigest()
    with Image.open(path) as im:
        size = im.size                      # (w, h) originali
        im = im.convert("RGB")
        ph = imagehash.phash(im, hash_size=HASH_SIZE)
    bits = np.packbits(ph.hash.flatten())   # 32 byte
    return str(path), sha, size, bits

def describe_all(files, desc):
    with ProcessPoolExecutor() as ex:
        out = list(tqdm(ex.map(describe, files, chunksize=16), total=len(files), desc=desc))
    return pd.DataFrame(out, columns=["path", "sha1", "size", "bits"])

if CACHE.exists():
    df_soco, df_sub = pickle.load(open(CACHE, "rb"))
else:
    df_soco = describe_all(soco_files, "hash SOCO")
    df_sub  = describe_all(sub_files,  "hash sub-SOCO")
    pickle.dump((df_soco, df_sub), open(CACHE, "wb"))
df_sub["split"] = df_sub.path.str.contains("/train/").map({True: "train", False: "test"})

In [4]:
# 1) match esatto sui byte
sha2idx = df_soco.groupby("sha1").indices
exact = {}
for i, s in tqdm(enumerate(df_sub.sha1), total=len(df_sub), desc="sha1 match"):
    if s in sha2idx:
        exact[i] = int(sha2idx[s][0])
print(f"match esatti: {len(exact)}/{len(df_sub)}")

sha1 match:   0%|          | 0/646 [00:00<?, ?it/s]

match esatti: 646/646


In [5]:
# 2) fallback pHash per i residui (stessa dimensione richiesta)
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
soco_bits = np.stack(df_soco.bits.values)            # (M, 32)
soco_size = np.array(df_soco["size"].tolist())       # (M, 2)

phash_match, rows = {}, []
for i in tqdm([i for i in range(len(df_sub)) if i not in exact], desc="phash match"):
    b = df_sub.bits.iloc[i]; sz = np.array(df_sub["size"].iloc[i])
    d = POP[np.bitwise_xor(soco_bits, b)].sum(1).astype(int)
    d[(soco_size != sz).any(1)] = 10_000             # dimensioni diverse -> escluso
    order = np.argsort(d)[:2]
    best, second = order[0], order[1]
    rows.append((i, int(best), d[best], d[second]))
    if d[best] <= PHASH_THR:
        phash_match[i] = int(best)

df_ph = pd.DataFrame(rows, columns=["sub_idx", "soco_idx", "dist", "dist_second"])
print(f"match pHash: {len(phash_match)}/{len(rows)} residui")
if len(df_ph):
    print(df_ph.dist.describe())
    print("ambigui (best e second vicini):", (df_ph.dist_second - df_ph.dist < 4).sum())

phash match: 0it [00:00, ?it/s]

match pHash: 0/0 residui


In [6]:
# 3) risultato finale
mapping = {**exact, **phash_match}
unmatched = [i for i in range(len(df_sub)) if i not in mapping]
print(f"mappate {len(mapping)}/{len(df_sub)}; non mappate: {len(unmatched)}")
print("collisioni (due immagini sub -> stessa SOCO):", len(mapping) - len(set(mapping.values())))

df_map = pd.DataFrame({
    "sub_path":  [df_sub.path[i] for i in mapping],
    "split":     [df_sub.split[i] for i in mapping],
    "soco_path": [df_soco.path[j] for j in mapping.values()],
    "how":       ["sha1" if i in exact else "phash" for i in mapping],
})
df_map.to_csv(ROOT / "sub_to_soco_mapping.csv", index=False)

used = set(mapping.values())
df_missing = df_soco[~df_soco.index.isin(used)][["path", "size"]]
df_missing.to_csv(ROOT / "soco_images_not_in_sub.csv", index=False)
print("immagini SOCO NON presenti in sub-SOCO:", len(df_missing))
display(df_map.how.value_counts())

mappate 646/646; non mappate: 0
collisioni (due immagini sub -> stessa SOCO): 0
immagini SOCO NON presenti in sub-SOCO: 3354


how
sha1    646
Name: count, dtype: int64

In [7]:
# 4) sanity check: da quali cartelle/categorie SOCO vengono train e test?
df_map["soco_cat"] = df_map.soco_path.map(lambda p: Path(p).relative_to(SOCO / "Images").parts[0])
display(pd.crosstab(df_map.soco_cat, df_map.split).head(20))

split,test,train
soco_cat,,
aeroplane,0,4
american_black_bear,19,2
arctic_fox,0,4
ax,0,11
bed,0,4
bench,0,4
bighorn,4,5
boston_bull,3,0
brown_bear,3,16


In [8]:
# 1) la submission copre tutte le query del test?
t = pd.read_csv(SUB / "test.csv")
s = pd.read_csv(SUB / "sample_submission.csv")
print(len(t), len(s), "stessi id:", set(t.id) == set(s.id))

# 2) struttura degli split originali di SOCO
TS = SOCO / "PairAnnotations" / "trainsplits"
for p in sorted(TS.rglob("*"))[:30]:
    print(p.relative_to(SOCO), p.stat().st_size if p.is_file() else "")

577 577 stessi id: True
PairAnnotations/trainsplits/test 
PairAnnotations/trainsplits/test/aeroplane 
PairAnnotations/trainsplits/test/aeroplane/00001-aeroplane_2-aeroplane_40.json 3860
PairAnnotations/trainsplits/test/aeroplane/00002-aeroplane_25-aeroplane_9.json 5419
PairAnnotations/trainsplits/test/aeroplane/00003-aeroplane_11-aeroplane_34.json 5461
PairAnnotations/trainsplits/test/aeroplane/00004-aeroplane_10-aeroplane_32.json 4627
PairAnnotations/trainsplits/test/aeroplane/00005-aeroplane_26-aeroplane_28.json 6492
PairAnnotations/trainsplits/test/aeroplane/00006-aeroplane_13-aeroplane_35.json 4338
PairAnnotations/trainsplits/test/aeroplane/00007-aeroplane_30-aeroplane_31.json 5444
PairAnnotations/trainsplits/test/aeroplane/00008-aeroplane_1-aeroplane_33.json 6521
PairAnnotations/trainsplits/test/aeroplane/00009-aeroplane_37-aeroplane_4.json 5111
PairAnnotations/trainsplits/test/aeroplane/00010-aeroplane_27-aeroplane_29.json 5138
PairAnnotations/trainsplits/test/aeroplane/00011-aer

In [9]:
import re, json

# --- coppie del test (e train) originale di SOCO, dai nomi dei file ---
PAT = re.compile(r"^\d+-(.+)-(.+)\.json$")
def soco_pairs(split):
    out = []
    for f in tqdm(list((SOCO / "PairAnnotations" / "trainsplits" / split).rglob("*.json")), desc=f"pairs {split}"):
        m = PAT.match(f.name)
        a, b = m.groups()
        pa = SOCO / "Images" / a.rsplit("_", 1)[0] / f"{a}.JPEG"
        pb = SOCO / "Images" / b.rsplit("_", 1)[0] / f"{b}.JPEG"
        out.append((str(pa), str(pb)))
    return out

soco_test_pairs  = soco_pairs("test")
soco_train_pairs = soco_pairs("train")
soco_test_imgs   = {p for pr in soco_test_pairs for p in pr}
soco_train_imgs  = {p for pr in soco_train_pairs for p in pr}
print(f"SOCO originale: {len(soco_train_pairs)} coppie train, {len(soco_test_pairs)} coppie test; "
      f"{len(soco_train_imgs)} / {len(soco_test_imgs)} immagini")

# --- mappa sub-path (relativo) -> path SOCO ---
sub2soco = {str(Path(r.sub_path).relative_to(SUB)): r.soco_path for r in df_map.itertuples()}

sub_test_imgs  = {sub2soco[k] for k in sub2soco if k.startswith("images/test")}
sub_train_imgs = {sub2soco[k] for k in sub2soco if k.startswith("images/train")}

# --- coppie del tuo subset, espresse in path SOCO ---
def sub_pairs(js):
    d = json.load(open(SUB / js))
    return {(sub2soco[p["source_image"]], sub2soco[p["target_image"]]) for p in d["pairs"]}
sub_test_pairs  = sub_pairs("test_pairs.json")
sub_train_pairs = sub_pairs("train_pairs.json")

# --- controlli ---
print("coppie sub-test presenti nel test SOCO :", len(sub_test_pairs & set(soco_test_pairs)), "/", len(sub_test_pairs))
print("coppie sub-train presenti nel train SOCO:", len(sub_train_pairs & set(soco_train_pairs)), "/", len(sub_train_pairs))
print("immagini sub-test nel test SOCO         :", len(sub_test_imgs & soco_test_imgs), "/", len(sub_test_imgs))
print("immagini sub-train nel test SOCO (leak?):", len(sub_train_imgs & soco_test_imgs))
print("immagini sub-train nel train SOCO       :", len(sub_train_imgs & soco_train_imgs), "/", len(sub_train_imgs))

# --- quanto test originale NON è nel tuo subset ---
print("coppie test SOCO fuori dal tuo subset   :", len(set(soco_test_pairs) - sub_test_pairs))
print("immagini test SOCO fuori dal tuo subset :", len(soco_test_imgs - sub_test_imgs))

pairs test:   0%|          | 0/10000 [00:00<?, ?it/s]

pairs train:   0%|          | 0/10000 [00:00<?, ?it/s]

SOCO originale: 10000 coppie train, 10000 coppie test; 2000 / 1998 immagini
coppie sub-test presenti nel test SOCO : 8 / 98
coppie sub-train presenti nel train SOCO: 33 / 392
immagini sub-test nel test SOCO         : 67 / 129
immagini sub-train nel test SOCO (leak?): 257
immagini sub-train nel train SOCO       : 260 / 517
coppie test SOCO fuori dal tuo subset   : 9992
immagini test SOCO fuori dal tuo subset : 1931


In [10]:
# ordine invertito?
unordered = lambda S: {frozenset(p) for p in S}
print("sub-test, ignorando la direzione:", len(unordered(sub_test_pairs) & unordered(soco_test_pairs)), "/", len(sub_test_pairs))

# da dove vengono le coppie? guarda cross/intra
for d in ("cross", "intra"):
    base = SOCO / "PairAnnotations" / d
    files = list(base.rglob("*.json"))
    print(d, len(files), [f.relative_to(base).as_posix() for f in files[:2]])

sub-test, ignorando la direzione: 12 / 98
cross 20000 ['jinrikisha/00144-jinrikisha_18-jinrikisha_23.json', 'jinrikisha/00095-cart_25-jinrikisha_13.json']
intra 20000 ['jinrikisha/00033-jinrikisha_11-jinrikisha_1.json', 'jinrikisha/00151-jinrikisha_9-jinrikisha_16.json']


In [11]:
def pairs_from(folder):
    out = {}
    for f in tqdm(list((SOCO / "PairAnnotations" / folder).rglob("*.json")), desc=folder):
        a, b = PAT.match(f.name).groups()
        pa = str(SOCO / "Images" / a.rsplit("_", 1)[0] / f"{a}.JPEG")
        pb = str(SOCO / "Images" / b.rsplit("_", 1)[0] / f"{b}.JPEG")
        out[(pa, pb)] = f
    return out

intra, cross = pairs_from("intra"), pairs_from("cross")
print("intra ⊇ trainsplits?", len((set(soco_train_pairs) | set(soco_test_pairs)) & set(intra)), "/ 20000")

U = lambda S: {frozenset(p) for p in S}
for name, sub in [("sub-test", sub_test_pairs), ("sub-train", sub_train_pairs)]:
    in_intra = len(U(sub) & U(intra)); in_cross = len(U(sub) & U(cross))
    in_any   = len(U(sub) & (U(intra) | U(cross)))
    print(f"{name}: {len(sub)} coppie | intra {in_intra} | cross {in_cross} | almeno una {in_any}")

# categoria sorgente vs target delle coppie del tuo test: quante sono cross-category?
cat = lambda p: Path(p).parent.name
print("coppie sub-test con categorie diverse:", sum(cat(a) != cat(b) for a, b in sub_test_pairs), "/", len(sub_test_pairs))

intra:   0%|          | 0/20000 [00:00<?, ?it/s]

cross:   0%|          | 0/20000 [00:00<?, ?it/s]

intra ⊇ trainsplits? 3336 / 20000
sub-test: 98 coppie | intra 98 | cross 2 | almeno una 98
sub-train: 392 coppie | intra 392 | cross 35 | almeno una 392
coppie sub-test con categorie diverse: 0 / 98


In [12]:
t.head()

,id,pair_id,source_image,target_image,query_index,source_x,source_y
0,test_q_00341,test_pair_0057,images/test/img_0554.jpeg,images/test/img_0103.jpeg,3,276.580645,189.368664
1,test_q_00212,test_pair_0035,images/test/img_0582.jpeg,images/test/img_0046.jpeg,1,192.000000,143.000000
2,test_q_00426,test_pair_0072,images/test/img_0253.jpeg,images/test/img_0618.jpeg,9,102.000000,534.000000
3,test_q_00240,test_pair_0041,images/test/img_0433.jpeg,images/test/img_0082.jpeg,3,162.000000,247.000000
4,test_q_00102,test_pair_0015,images/test/img_0433.jpeg,images/test/img_0124.jpeg,7,277.000000,162.000000


In [13]:
s.head()

,id,prediction
0,test_q_00341,"[250.0,167.5]"
1,test_q_00212,"[250.0,187.5]"
2,test_q_00426,"[320.0,240.0]"
3,test_q_00240,"[225.0,220.0]"
4,test_q_00102,"[100.0,97.0]"


In [14]:
import ast, json
from collections import defaultdict

# ---------- 1) indice (img_a, img_b) -> (json, invertita?) ----------
idx = {}
for swapped in (True, False):          # le coppie nella direzione giusta sovrascrivono le invertite
    for d in (cross, intra):           # intra sovrascrive cross
        for (a, b), f in d.items():
            idx[(b, a) if swapped else (a, b)] = (f, swapped)

# ---------- 2) ground truth per ogni query del test ----------
tp = json.load(open(SUB / "test_pairs.json"))["pairs"]
rows, pair_stats = [], []
for p in tp:
    sa, sb = sub2soco[p["source_image"]], sub2soco[p["target_image"]]
    f, swapped = idx[(sa, sb)]
    J = json.load(open(f))
    s_key, t_key = ("trg", "src") if swapped else ("src", "trg")
    skps, tkps = J[f"{s_key}_keypoints"], J[f"{t_key}_keypoints"]
    tpos = {k["name"]: k["pos"] for k in tkps}
    by_concept = defaultdict(list)
    for k in tkps: by_concept[k["concept_id"]].append(k["pos"])
    spos = np.array([k["pos"] for k in skps])

    n_overlap = len(set(tpos) & {k["name"] for k in skps})
    pair_stats.append((p["pair_id"], len(p["source_xy"]), n_overlap, swapped, f.parent.parent.name))

    for i, (x, y) in enumerate(p["source_xy"]):
        d = np.hypot(*(spos - [x, y]).T)
        j = int(d.argmin()); k = skps[j]
        gt, how = None, "none"
        if k["name"] in tpos:
            gt, how = tpos[k["name"]], "name"
        elif len(by_concept[k["concept_id"]]) == 1:
            gt, how = by_concept[k["concept_id"]][0], "concept"
        rows.append(dict(pair_id=p["pair_id"], i=i, sx=x, sy=y, dist=d[j], kp=k["name"],
                         how=how, gx=gt[0] if gt else np.nan, gy=gt[1] if gt else np.nan,
                         t_w=p["target_size"][0], t_h=p["target_size"][1],
                         bbox=J["src_bndbox" if swapped else "trg_bndbox"]))
gt_df = pd.DataFrame(rows)
ps = pd.DataFrame(pair_stats, columns=["pair_id", "n_query_sub", "n_overlap_soco", "swapped", "folder"])

# ---------- 3) agganciare le righe di test.csv ----------
# query_index è 0- o 1-based? lo verifichiamo sulle coordinate
for off in (0, 1):
    m = t.merge(gt_df, left_on=["pair_id", t.query_index - off], right_on=["pair_id", "i"], how="left")
    print(f"offset {off}: coordinate coerenti =", np.isclose(m.source_x, m.sx, atol=1e-3).mean())

offset 0: coordinate coerenti = 1.0
offset 1: coordinate coerenti = 0.0017331022530329288


In [15]:
OFF = 0
G = t.merge(gt_df, left_on=["pair_id", t.query_index - OFF], right_on=["pair_id", "i"], how="left")

# ---------- 4) risposta alla domanda: le query vengono tutte da SOCO? ----------
print("query test:", len(t), "| agganciate:", G.gx.notna().sum(), "| senza GT:", G.gx.isna().sum())
print("dist keypoint (max):", G.dist.max(), "| query con dist > 0.5px:", (G.dist > 0.5).sum())
print(G.how.value_counts())
print("swapped:", ps.swapped.value_counts().to_dict(), "| cartella:", ps.folder.value_counts().to_dict())
print("query subset vs keypoint in overlap SOCO:\n", (ps.n_query_sub - ps.n_overlap_soco).describe())

# ---------- 5) valutazione di una submission ----------
def evaluate(sub_csv, alphas=(0.05, 0.1, 0.15)):
    s = pd.read_csv(sub_csv)
    xy = np.array(s.prediction.map(ast.literal_eval).tolist(), dtype=float)
    s["px"], s["py"] = xy[:, 0], xy[:, 1]
    e = G[["id", "gx", "gy", "t_w", "t_h", "bbox"]].merge(s[["id", "px", "py"]], on="id", how="left")
    e = e.dropna(subset=["gx", "px"])
    err = np.hypot(e.px - e.gx, e.py - e.gy)
    bb = np.array(e.bbox.tolist())
    norm_img = np.maximum(e.t_w, e.t_h)
    norm_bb = np.maximum(bb[:, 2] - bb[:, 0], bb[:, 3] - bb[:, 1])
    out = {"n_valutate": len(e), "err_medio_px": err.mean()}
    for a in alphas:
        out[f"PCK@{a}_img"] = (err <= a * norm_img).mean()
        out[f"PCK@{a}_bbox"] = (err <= a * norm_bb).mean()
    return pd.Series(out)

display(evaluate(SUB / "sample_submission.csv"))

query test: 577 | agganciate: 535 | senza GT: 42
dist keypoint (max): 0.0 | query con dist > 0.5px: 0
how
name       429
concept    106
none        42
Name: count, dtype: int64
swapped: {False: 98} | cartella: {'intra': 98}
query subset vs keypoint in overlap SOCO:
 count    98.000000
mean      4.785714
std       2.829338
min       0.000000
25%       3.000000
50%       5.000000
75%       6.750000
max      12.000000
dtype: float64


n_valutate       535.000000
err_medio_px     126.921531
PCK@0.05_img       0.035514
PCK@0.05_bbox      0.001869
PCK@0.1_img        0.171963
PCK@0.1_bbox       0.007477
PCK@0.15_img       0.278505
PCK@0.15_bbox      0.020561
dtype: float64

In [16]:
# 1) quanto è grande il bbox rispetto all'immagine target?
e = G.dropna(subset=["gx"])
bb = np.array(e.bbox.tolist())
print((np.maximum(bb[:,2]-bb[:,0], bb[:,3]-bb[:,1]) / np.maximum(e.t_w, e.t_h)).describe())

# 2) chi sono le 42 query senza GT: quanti candidati per concetto ha il target?
pp = {p["pair_id"]: p for p in tp}
def diag(r):
    p = pp[r.pair_id]
    f, _ = idx[(sub2soco[p["source_image"]], sub2soco[p["target_image"]])]
    J = json.load(open(f))
    k = next(k for k in J["src_keypoints"] if k["name"] == r.kp)
    c = [x["name"] for x in J["trg_keypoints"] if x["concept_id"] == k["concept_id"]]
    return pd.Series(dict(concept=k["concept_name"], n_cand=len(c), cand=c))
none = G[G.how == "none"].apply(diag, axis=1)
print(none.n_cand.value_counts())
print(none.head(10))

# 3) test sottocampionato?
print("source_xy totali:", sum(len(p["source_xy"]) for p in tp), "vs query test:", len(t))

count    535.000000
mean       0.352223
std        0.271579
min        0.065312
25%        0.136620
50%        0.215600
75%        0.555500
max        0.997938
dtype: float64
n_cand
2    38
3     4
Name: count, dtype: int64
                concept  n_cand  \
14                  paw       2   
29                stand       3   
30   vehicle_front_wing       3   
34         wheel_center       2   
38                  paw       2   
41                  paw       2   
75                  paw       2   
102                 paw       2   
114             ski_tip       2   
125                 paw       2   

                                                  cand  
14                   [right_back_paw, right_front_paw]  
29   [stand_bottom_back_left, stand_bottom_front_le...  
30   [f1_front_wing_left_end, f1_front_wing_left_ti...  
34   [bike_wheel_front_center, wheel_side_left_center]  
38                   [right_back_paw, right_front_paw]  
41                     [left_back_paw, right_bac

In [17]:
# ---- 1) per ogni punto sorgente (898) calcolo candidati target ----
rows = []
for p in tp:
    f, swapped = idx[(sub2soco[p["source_image"]], sub2soco[p["target_image"]])]
    J = json.load(open(f))
    skps, tkps = J["src_keypoints"], J["trg_keypoints"]
    spos = np.array([k["pos"] for k in skps])
    for i, (x, y) in enumerate(p["source_xy"]):
        k = skps[int(np.hypot(*(spos - [x, y]).T).argmin())]
        cands = [c["pos"] for c in tkps if c["concept_id"] == k["concept_id"]]
        named = [c["pos"] for c in tkps if c["name"] == k["name"]]
        rows.append(dict(pair_id=p["pair_id"], i=i, kp=k["name"], cands=cands,
                         named=named[0] if named else None,
                         t_w=p["target_size"][0], t_h=p["target_size"][1],
                         bbox=J["trg_bndbox"]))
Q = pd.DataFrame(rows)
kept = set(zip(t.pair_id, t.query_index))
Q["kept"] = [(a, b) in kept for a, b in zip(Q.pair_id, Q.i)]
Q["has_concept"] = Q.cands.map(len) > 0
Q["has_name"] = Q.named.notna()
Q["n_cand"] = Q.cands.map(len)

print(pd.crosstab(Q.has_concept, Q.kept))
print(pd.crosstab(Q.has_name, Q.kept))
print(pd.crosstab(Q.n_cand, Q.kept))

kept         False  True 
has_concept              
False          321      0
True             0    577
kept      False  True 
has_name              
False       321    148
True          0    429
kept    False  True 
n_cand              
0         321      0
1           0    274
2           0    255
3           0     20
4           0     28


In [18]:
G2 = t.merge(Q, left_on=["pair_id", "query_index"], right_on=["pair_id", "i"], how="left")

def evaluate(sub_csv, alphas=(0.05, 0.1, 0.15, 0.2)):
    s = pd.read_csv(sub_csv)
    xy = np.array(s.prediction.map(ast.literal_eval).tolist(), dtype=float)
    s["px"], s["py"] = xy[:, 0], xy[:, 1]
    e = G2.merge(s[["id", "px", "py"]], on="id", how="left")
    assert e.px.notna().all(), f"{e.px.isna().sum()} query senza predizione"
    P = e[["px", "py"]].to_numpy()
    # errore minimo sui candidati del concetto (variante estesa, many-to-many)
    e["err_min"] = [min(np.hypot(*(np.array(c) - p)) for c in cs) for cs, p in zip(e.cands, P)]
    # errore sul solo omonimo (variante stretta)
    e["err_name"] = [np.hypot(*(np.array(n) - p)) if n is not None else np.nan
                     for n, p in zip(e.named, P)]
    bb = np.array(e.bbox.tolist())
    norms = {"img": np.maximum(e.t_w, e.t_h).to_numpy(),
             "bbox": np.maximum(bb[:, 2] - bb[:, 0], bb[:, 3] - bb[:, 1])}
    out = {}
    for nn, nv in norms.items():
        for a in alphas:
            out[f"PCK@{a} {nn} | min-candidati (n={len(e)})"] = (e.err_min <= a * nv).mean()
            m = e.err_name.notna()
            out[f"PCK@{a} {nn} | solo-nome (n={m.sum()})"] = (e.err_name[m] <= a * nv[m]).mean()
    out["err medio px (min)"] = e.err_min.mean()
    return pd.Series(out)

display(evaluate(SUB / "sample_submission.csv"))

PCK@0.05 img | min-candidati (n=577)       0.050260
PCK@0.05 img | solo-nome (n=429)           0.041958
PCK@0.1 img | min-candidati (n=577)        0.173310
PCK@0.1 img | solo-nome (n=429)            0.174825
PCK@0.15 img | min-candidati (n=577)       0.270364
PCK@0.15 img | solo-nome (n=429)           0.265734
PCK@0.2 img | min-candidati (n=577)        0.464471
PCK@0.2 img | solo-nome (n=429)            0.426573
PCK@0.05 bbox | min-candidati (n=577)      0.003466
PCK@0.05 bbox | solo-nome (n=429)          0.002331
PCK@0.1 bbox | min-candidati (n=577)       0.008666
PCK@0.1 bbox | solo-nome (n=429)           0.009324
PCK@0.15 bbox | min-candidati (n=577)      0.025997
PCK@0.15 bbox | solo-nome (n=429)          0.023310
PCK@0.2 bbox | min-candidati (n=577)       0.043328
PCK@0.2 bbox | solo-nome (n=429)           0.041958
err medio px (min)                       121.524517
dtype: float64

In [19]:
def pck(sub_csv, alpha=0.10, clip=False, verbose=True):
    s = pd.read_csv(sub_csv)
    assert list(s.columns) == ["id", "prediction"], "colonne errate"
    assert s.id.is_unique and set(s.id) == set(t.id), "id mancanti/extra/duplicati"
    xy = np.array(s.prediction.map(json.loads).tolist(), dtype=float)   # JSON array [x,y]
    assert xy.shape == (len(s), 2) and np.isfinite(xy).all(), "predizioni non valide"
    s["px"], s["py"] = xy[:, 0], xy[:, 1]

    e = G2.merge(s[["id", "px", "py"]], on="id", how="left")
    bb = np.array(e.bbox.tolist(), dtype=float)
    if clip:  # bbox ritagliato ai bordi dell'immagine target
        bb[:, [0, 2]] = bb[:, [0, 2]].clip(0, e.t_w.to_numpy()[:, None])
        bb[:, [1, 3]] = bb[:, [1, 3]].clip(0, e.t_h.to_numpy()[:, None])
    L = np.maximum(bb[:, 2] - bb[:, 0], bb[:, 3] - bb[:, 1])

    P = e[["px", "py"]].to_numpy()
    d = np.array([min(np.hypot(*(np.array(c) - p)) for c in cs) for cs, p in zip(e.cands, P)])
    correct = d / L <= alpha          # "falls within a circle" -> <=
    if verbose:
        print(f"PCK@{alpha:.2f} = {correct.mean():.4f}  ({correct.sum()}/{len(e)})  "
              f"[bbox {'clippato' if clip else 'originale'}]")
    return correct.mean()

pck(SUB / "sample_submission.csv")
pck(SUB / "sample_submission.csv", clip=True)
# la tua submission:
pck("submission-F-Ensamble.csv")

PCK@0.10 = 0.0087  (5/577)  [bbox originale]
PCK@0.10 = 0.0087  (5/577)  [bbox clippato]
PCK@0.10 = 0.7331  (423/577)  [bbox originale]


np.float64(0.733102253032929)

In [20]:
pck("submission-F74.csv")

PCK@0.10 = 0.7331  (423/577)  [bbox originale]


np.float64(0.733102253032929)

In [21]:
pck("M-submission.csv")

PCK@0.10 = 0.7262  (419/577)  [bbox originale]


np.float64(0.7261698440207972)

In [25]:
pck("submission_four-recipes-avg_object-zoom.csv")

PCK@0.10 = 0.7140  (412/577)  [bbox originale]


np.float64(0.7140381282495667)

In [22]:
def load(p):
    s = pd.read_csv(p)
    xy = np.array(s.prediction.map(json.loads).tolist(), dtype=float)
    return s.assign(px=xy[:, 0], py=xy[:, 1]).set_index("id").loc[t.id]

a, b = load("submission-F74.csv"), load("submission-F-Ensamble.csv")
diff = np.hypot(a.px - b.px, a.py - b.py)
print("file identici:", a.prediction.equals(b.prediction))
print("righe con predizione diversa:", (diff > 1e-6).sum(), "/", len(diff))
print(diff.describe())

def margin(s):
    e = G2.set_index("id").loc[t.id]
    bb = np.array(e.bbox.tolist(), dtype=float)
    L = np.maximum(bb[:, 2]-bb[:, 0], bb[:, 3]-bb[:, 1])
    P = s[["px", "py"]].to_numpy()
    d = np.array([min(np.hypot(*(np.array(c)-p)) for c in cs) for cs, p in zip(e.cands, P)])
    return d / L

ra, rb = margin(a), margin(b)
print("query con esito diverso:", ((ra <= .1) != (rb <= .1)).sum())
for k in (0.05, 0.10, 0.15, 0.20):
    print(f"PCK@{k}: F74={np.mean(ra<=k):.4f}  Ensemble={np.mean(rb<=k):.4f}")

file identici: False
righe con predizione diversa: 577 / 577
count    577.000000
mean       9.258498
std       29.541032
min        0.029892
25%        0.718437
50%        1.680062
75%        4.807590
max      294.188361
dtype: float64


query con esito diverso: 36
PCK@0.05: F74=0.4697  Ensemble=0.4541
PCK@0.1: F74=0.7331  Ensemble=0.7331
PCK@0.15: F74=0.8423  Ensemble=0.8198
PCK@0.2: F74=0.8804  Ensemble=0.8718


In [23]:
def pck_var(p, cand_mode="min", clip=False, alpha=0.10):
    s = load(p); e = G2.set_index("id").loc[t.id]
    bb = np.array(e.bbox.tolist(), float)
    if clip:
        bb[:, [0, 2]] = bb[:, [0, 2]].clip(0, e.t_w.to_numpy()[:, None])
        bb[:, [1, 3]] = bb[:, [1, 3]].clip(0, e.t_h.to_numpy()[:, None])
    L = np.maximum(bb[:, 2]-bb[:, 0], bb[:, 3]-bb[:, 1])
    P = s[["px", "py"]].to_numpy()
    d = []
    for cs, nm, pt in zip(e.cands, e.named, P):
        if cand_mode == "name" and nm is not None:
            cs = [nm]
        d.append(min(np.hypot(*(np.array(c)-pt)) for c in cs))
    return np.mean(np.array(d)/L <= alpha)

for f in ("submission-F74.csv", "submission-F-Ensamble.csv"):
    for cm in ("min", "name"):
        for cl in (False, True):
            print(f, cm, "clip" if cl else "raw", round(pck_var(f, cm, cl), 4))

submission-F74.csv min raw 0.7331
submission-F74.csv min clip 0.7331
submission-F74.csv name raw 0.5737
submission-F74.csv name clip 0.5737
submission-F-Ensamble.csv min raw 0.7331
submission-F-Ensamble.csv min clip 0.7331
submission-F-Ensamble.csv name raw 0.5702
submission-F-Ensamble.csv name clip 0.5702


In [24]:
r74, rens = margin(a), margin(b)
ok74, okens = (r74 <= .1), (rens <= .1)
print("F74 giuste:", ok74.sum(), "| Ens giuste:", okens.sum())
print("solo F74:", (ok74 & ~okens).sum(), "| solo Ens:", (~ok74 & okens).sum())
# quanto è probabile vedere +2 su 101 per caso, dati i 36 esiti discordanti?
rng = np.random.default_rng(0)
disc = np.flatnonzero(ok74 != okens)
res = []
for _ in range(20000):
    sub = rng.choice(len(t), 101, replace=False)
    res.append(ok74[sub].sum() - okens[sub].sum())
res = np.array(res)
print("differenza su 101 query: media", res.mean(), "| P(|diff|>=2) =", (np.abs(res) >= 2).mean())

F74 giuste: 423 | Ens giuste: 423
solo F74: 18 | solo Ens: 18
differenza su 101 query: media -0.02275 | P(|diff|>=2) = 0.50525
